# Opdracht - 


## Inlezen van de variabelen - creëren van de omgeving

In [1]:
import os
import json
import sys
import pandas as pd
from datetime import date, timedelta, datetime
import calendar


os.chdir(r"C:\Users\kurtm\Documents\Oefening01.01")

## Main programma - lees de broninformatie in

In [17]:
# Lees de configuratie in
config = load_project_config(r"./config.json")


#Overloop alle files in de Source directory
SourceDir = os.path.join(config["Directories"]["Root"],config["Directories"]["Data"],config["Directories"]["Input"])

All_df = pd.DataFrame()

for ReadFile in os.listdir(SourceDir):
    # lees de file in
    if  not ReadFile.endswith('.csv'):
        continue
    EchteGeboortedag = os.path.splitext(ReadFile)[0] 
    #bepaal of geldige datum in filenaam - check dan in de config wat je moet doen
    if not is_geldige_datum(EchteGeboortedag):
        functie_naam = config["Decisions"]["Wrong Date"]
        if not functie_naam in globals() or callable(globals()[functie_naam]):
            continue
        EchteGeboortedag = globals()[functie_naam](EchteGeboortedag)


    File_df = pd.read_csv(os.path.join(SourceDir,ReadFile))
    File_df['verwachte datum'] = pd.to_datetime(File_df['verwachte datum'])
    File_df["Echte geboortedag"] = pd.to_datetime(EchteGeboortedag)
    File_df["SourceFile"] = ReadFile
    All_df = pd.concat([All_df, File_df], ignore_index=True)

All_df = All_df.sort_values('Echte geboortedag')
All_df.to_excel(os.path.join(config["Directories"]["Root"],config["Directories"]["Data"],config["Directories"]["Temp"],config["Files"]["RestFile"]), index=False)


In [16]:
def is_geldige_datum(datum: str) -> bool:
    try:
        datetime.strptime(datum, '%Y-%m-%d')
        return True
    except ValueError:
        return False

def volgende_geldige_datum(datum: str) -> str:
    jaar, maand, dag = map(int, datum.split('-'))
    try:
        return date(jaar, maand, dag).strftime('%Y-%m-%d')
    except ValueError:
        if maand == 12:
            maand = 1
            jaar += 1
        else:
            maand += 1
        return date(jaar, maand, 1).strftime('%Y-%m-%d')

def laatste_geldige_datum(datum: str) -> str:
    jaar, maand, dag = map(int, datum.split('-'))
    laatste_dag = calendar.monthrange(jaar, maand)[1]
    geldige_dag = min(dag, laatste_dag)
    return date(jaar, maand, geldige_dag).strftime('%Y-%m-%d')

In [8]:
# inlezen configfile

def load_project_config(config_path):
    """
    Laadt de project-configuratie uit een JSON-bestand.
    Stopt het programma met een duidelijke foutmelding als er iets mis is.
    
    Returns:
        dict: de geladen configuratie
        
    Exits:
        sys.exit(1) bij file not found
        sys.exit(2) bij ongeldige JSON
        sys.exit(3) bij onverwachte andere fouten
    """
    # Zorg dat we een absoluut pad hebben (veiliger in notebooks/scripts)
    config_file = os.path.abspath(config_path)

    # 1. Bestaat het bestand überhaupt?
    if not os.path.isfile(config_file):
        print("FOUT: Configuratiebestand niet gevonden", file=sys.stderr)
        print(f"   Pad: {config_file}", file=sys.stderr)
        print("   Verwacht: JSON-bestand met 'directories' sleutel", file=sys.stderr)
        sys.exit(1)

    # 2. Proberen in te lezen
    try:
        with open(config_file, "r", encoding="utf-8") as f:
            config = json.load(f)

    except json.JSONDecodeError as e:
        print("FOUT: Ongeldige JSON syntax in configuratiebestand", file=sys.stderr)
        print(f"Bestand  : {config_file}", file=sys.stderr)
        print(f"Regel    : {e.lineno}", file=sys.stderr)
        print(f"Positie  : {e.colno}", file=sys.stderr)
        print(f"Boodschap: {e.msg}", file=sys.stderr)
        sys.exit(2)

    except Exception as e:
        print("FOUT: Kan configuratiebestand niet lezen", file=sys.stderr)
        print(f"Bestand : {config_file}", file=sys.stderr)
        print(f"Fout    : {type(e).__name__}", file=sys.stderr)
        print(f"Bericht : {e}", file=sys.stderr)
        sys.exit(3)

    # 3. Minimale inhoudscontrole
    if not isinstance(config, dict):
        print("FOUT: Configuratie moet een JSON object (dictionary) zijn", file=sys.stderr)
        sys.exit(4)

    if "Directories" not in config:
        print("FOUT: Verplichte sleutel 'Directories' ontbreekt in configuratie", file=sys.stderr)
        sys.exit(5)

    if not isinstance(config["Directories"], dict):
        print("FOUT: 'Directories' moet een object (dictionary) zijn", file=sys.stderr)
        sys.exit(6)

    

    # Alles ok → geef config terug
    return config